# SFT Training Launcher

Colab 런타임에서 `scripts/train_sft.py`를 실행하기 위한 얇은 런처 노트북입니다.

GitHub repo와 실험 설정입니다. Colab에서 실행하기 전에 이 셀의 값만 수정하세요.

In [ ]:
GITHUB_REPO_URL = "https://github.com/c-peace/LEET_LLM_PEFT.git"
GITHUB_BRANCH = "main"
PROJECT_DIR = "/content/seonji-llm"

HF_USERNAME = "choipeace"
PROJECT_NAME = "LEET_seonji"
EXPERIMENT_VERSION = "v1"
EXPERIMENT_TITLE = "baseline"
EXPERIMENT_HYPOTHESIS = """기본 SFT 설정으로 선지 생성 품질이 개선되는지 확인한다."""
HF_REPO_PRIVATE = True

OUTPUT_ROOT = "/content/sft_outputs/runs"
BASE_CONFIG_PATH = "configs/train_config_v1.json"
TMP_CONFIG_PATH = "/content/train_config_no_drive.json"

GitHub에서 프로젝트 코드를 가져오고 작업 폴더로 이동합니다.

In [ ]:
import os
import subprocess
from pathlib import Path

if "YOUR_GITHUB_ID" in GITHUB_REPO_URL:
    raise ValueError("GITHUB_REPO_URL을 본인 GitHub repo 주소로 바꾼 뒤 실행하세요.")

project_dir = Path(PROJECT_DIR)
if project_dir.exists():
    subprocess.run(["git", "-C", str(project_dir), "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(project_dir)], check=True)

os.chdir(project_dir)
print(f"cwd={Path.cwd()}")
for required_path in ["scripts/train_sft.py", "scripts/upload_hf_adapter.py", "sft_dataset.json", "configs/train_config_v1.json"]:
    path = Path(required_path)
    if not path.exists():
        raise FileNotFoundError(f"required file not found: {path}")
    print(f"found: {path}")

런타임 GPU를 확인합니다.

In [ ]:
!nvidia-smi

필요 패키지를 설치합니다.

In [ ]:
!pip install -U transformers datasets peft accelerate bitsandbytes huggingface_hub

Hugging Face Hub에 로그인합니다. 토큰은 노트북 입력창에 직접 붙여넣고, 코드나 채팅에 남기지 마세요.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

Drive 용량을 쓰지 않도록 학습 결과 저장 위치를 Colab 임시 디스크로 바꿉니다.

In [ ]:
import json
from pathlib import Path

config_path = Path(BASE_CONFIG_PATH)
config = json.loads(config_path.read_text(encoding="utf-8"))
config["outputs"]["root_dir"] = OUTPUT_ROOT
config["training"]["save_strategy"] = "no"

tmp_config_path = Path(TMP_CONFIG_PATH)
tmp_config_path.write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"temporary config: {tmp_config_path}")
print(f"output root: {config['outputs']['root_dir']}")
print(f"save_strategy: {config['training']['save_strategy']}")
print(f"experiment: {PROJECT_NAME}-{EXPERIMENT_VERSION} / {EXPERIMENT_TITLE}")

Qwen baseline 학습 실행

In [ ]:
!python scripts/train_sft.py \
  --config {TMP_CONFIG_PATH} \
  --model_name Qwen/Qwen3.5-4B \
  --copy_config

# smoke test
# !python scripts/train_sft.py \
#   --config {TMP_CONFIG_PATH} \
#   --model_name Qwen/Qwen3.5-4B \
#   --run_id smoke_qwen \
#   --max_steps 10 \
#   --eval_sample_size 5 \
#   --copy_config

# !ls /content/sft_outputs/runs/smoke_qwen
# !cat /content/sft_outputs/runs/smoke_qwen/eval_summary.json


Gemma baseline 학습 실행

In [ ]:
!python scripts/train_sft.py \
  --config {TMP_CONFIG_PATH} \
  --model_name google/gemma-4-E4B-it \
  --copy_config

EXAONE baseline 학습 실행

In [ ]:
!python scripts/train_sft.py \
  --config {TMP_CONFIG_PATH} \
  --model_name LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct \
  --copy_config

최근 결과 폴더와 평가 요약을 확인합니다.

In [ ]:
import json
from pathlib import Path

runs = sorted(Path(OUTPUT_ROOT).glob("*"), key=lambda p: p.stat().st_mtime, reverse=True)
for run in runs[:5]:
    print(run)

if runs:
    latest_run = runs[0]
    summary_path = latest_run / "eval_summary.json"
    print(f"\nlatest_run={latest_run}")
    if summary_path.exists():
        print(json.dumps(json.loads(summary_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


최신 학습 결과를 Hugging Face Hub에 업로드합니다. 자세한 업로드 로직은 `scripts/upload_hf_adapter.py`에 있습니다.

In [ ]:
import subprocess

if HF_USERNAME == "YOUR_HF_USERNAME":
    raise ValueError("HF_USERNAME을 본인 Hugging Face 아이디로 바꾼 뒤 실행하세요.")

subprocess.run(
    [
        "python",
        "scripts/upload_hf_adapter.py",
        "--hf_username",
        HF_USERNAME,
        "--project_name",
        PROJECT_NAME,
        "--experiment_version",
        EXPERIMENT_VERSION,
        "--experiment_title",
        EXPERIMENT_TITLE,
        "--experiment_hypothesis",
        EXPERIMENT_HYPOTHESIS,
        "--runs_root",
        OUTPUT_ROOT,
        "--private",
        str(HF_REPO_PRIVATE).lower(),
    ],
    check=True,
)

다른 컴퓨터나 다른 Colab에서 adapter를 불러오는 예시입니다.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = "Qwen/Qwen3.5-4B"
adapter_repo = "choipeace/leet-seonji-qwen35-4b-v1"

tokenizer = AutoTokenizer.from_pretrained(adapter_repo)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, adapter_repo)

In [ ]:
from google.colab import runtime

runtime.unassign()
